In [0]:
CREATE CATALOG IF NOT EXISTS health;
USE CATALOG health;

CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;
CREATE SCHEMA IF NOT EXISTS ml;
CREATE SCHEMA IF NOT EXISTS perf;

CREATE VOLUME IF NOT EXISTS health.raw.synthea;

In [0]:
LIST '/Volumes/health/raw/synthea/batch_01/';

SELECT 'patients' AS file, count(*) AS row_count FROM read_files('/Volumes/health/raw/synthea/batch_01/patients.csv', format => 'csv', header => true)
UNION ALL
SELECT 'encounters', count(*) FROM read_files('/Volumes/health/raw/synthea/batch_01/encounters.csv', format => 'csv', header => true)
UNION ALL
SELECT 'observations', count(*) FROM read_files('/Volumes/health/raw/synthea/batch_01/observations.csv', format => 'csv', header => true);

In [0]:
SELECT 'patients' AS tbl, count(*) AS n FROM health.bronze.patients
UNION ALL SELECT 'encounters', count(*) FROM health.bronze.encounters
UNION ALL SELECT 'observations', count(*) FROM health.bronze.observations;

In [0]:
SELECT 'patients' AS tbl, count(*) FROM health.bronze.patients
UNION ALL SELECT 'encounters', count(*) FROM health.bronze.encounters
UNION ALL SELECT 'observations', count(*) FROM health.bronze.observations;

SELECT resource_type, count(*) AS n
FROM health.bronze.fhir_resources
GROUP BY resource_type ORDER BY n DESC;

In [0]:
SELECT 'encounters' AS tbl, count(*) FROM health.silver.encounters
UNION ALL SELECT 'observations', count(*) FROM health.silver.observations
UNION ALL SELECT 'dim_patient (current)', count(*) FROM health.silver.dim_patient WHERE __END_AT IS NULL
UNION ALL SELECT 'quarantine', count(*) FROM health.silver.quarantine_encounters
UNION ALL SELECT 'fhir_patients', count(*) FROM health.silver.fhir_patients;

SELECT encounter_class, count(*) AS n, round(avg(total_claim_cost), 2) AS avg_cost
FROM health.silver.encounters GROUP BY 1 ORDER BY n DESC;

In [0]:
SELECT patient_id, city, address, __START_AT, __END_AT
FROM health.silver.dim_patient
WHERE patient_id IN (SELECT patient_id FROM health.silver.dim_patient WHERE __END_AT IS NOT NULL)
ORDER BY patient_id, __START_AT
LIMIT 10;

SELECT
  count_if(__END_AT IS NULL)     AS current_rows,
  count_if(__END_AT IS NOT NULL) AS history_rows
FROM health.silver.dim_patient;